# Prompt Templates and Messages in LangChain V.1

> Converted from `04_prompt_templates_all.py` - part of **01 LangChain Foundations**.

## Setup

In [1]:
# ============ IMPORTS AND SETUP ===========================================
from langchain_core.prompts import (
    ChatPromptTemplate,
    FewShotChatMessagePromptTemplate,
    MessagesPlaceholder,
)
from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage,
)
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

True

### `demo_basic_templates`

Basic ChatPromptTemplate usage.

In [2]:
# ============ DEMO_BASIC_TEMPLATES ========================================
def demo_basic_templates():
    """Basic ChatPromptTemplate usage."""

    # Simple template
    simple = ChatPromptTemplate.from_template("Translate '{text}' to {language}")

    messages = simple.format_messages(text="Hello, world!", language="French")
    print("Simple template:")
    print(f"  {messages}")

    # Multi-message template
    multi = ChatPromptTemplate.from_messages(
        [
            ("system", "You are a translator. Be concise."),
            ("human", "Translate '{text}' to {language}"),
        ]
    )

    messages = multi.format_messages(text="Good morning", language="Japanese")
    print("\nMulti-message template:")
    for msg in messages:
        print(f"  {type(msg).__name__}: {msg.content}")

### `demo_message_types`

Working with different message types.

In [3]:
# ============ DEMO_MESSAGE_TYPES ==========================================
def demo_message_types():
    """Working with different message types."""

    model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    # Build conversation manually
    messages = [
        SystemMessage(content="You are a math tutor. Be brief."),
        HumanMessage(content="What's 5 * 5?"),
        AIMessage(content="25"),
        HumanMessage(content="And if I add 10?"),
    ]

    response = model.invoke(messages)
    print(f"Conversation result: {response.content}")

### `demo_messages_placeholder`

Use MessagesPlaceholder for dynamic conversation history.

In [4]:
# ============ DEMO_MESSAGES_PLACEHOLDER ===================================
def demo_messages_placeholder():
    """Use MessagesPlaceholder for dynamic conversation history."""

    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "You are a helpful assistant."),
            MessagesPlaceholder(variable_name="history"),
            ("human", "{question}"),
        ]
    )

    # Simulate conversation history
    history = [
        HumanMessage(content="My name is Paulo"),
        AIMessage(content="Nice to meet you, Paulo!"),
    ]

    messages = prompt.format_messages(history=history, question="What's my name?")

    print("With history placeholder:")
    for msg in messages:
        print(f"  {type(msg).__name__}: {msg.content[:50]}...")

    # Execute
    model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    
    chain = prompt | model
    response = chain.invoke({"history": history, "question": "What's my name?"})
    print(f"\nResponse: {response.content}")

### `demo_few_shot`

Few-shot prompting with examples.

In [5]:
# ============ DEMO_FEW_SHOT ===============================================
def demo_few_shot():
    """Few-shot prompting with examples."""

    # Define examples
    examples = [
        {"word": "happy", "opposite": "sad"},
        {"word": "fast", "opposite": "slow"},
        {"word": "big", "opposite": "small"},
    ]

    # Template for each example
    example_prompt = ChatPromptTemplate.from_messages(
        [
            ("human", "What's the opposite of '{word}'?"),
            ("ai", "The opposite of '{word}' is '{opposite}'."),
        ]
    )

    # Few-shot wrapper
    few_shot = FewShotChatMessagePromptTemplate(
        example_prompt=example_prompt,
        examples=examples,
    )

    # Final prompt
    final_prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "You give the opposite of words. Follow the examples."),
            few_shot,
            ("human", "What's the opposite of '{word}'?"),
        ]
    )

    # Test
    model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    chain = final_prompt | model

    response = chain.invoke({"word": "bright"})
    print(f"Few-shot result: {response.content}")

### `demo_prompt_composition`

Compose prompts from reusable parts.

In [6]:
# ============ DEMO_PROMPT_COMPOSITION =====================================
def demo_prompt_composition():
    """Compose prompts from reusable parts."""

    # Reusable system prompt
    persona = ChatPromptTemplate.from_messages(
        [("system", "You are a {role}. Your tone is {tone}.")]
    )

    # Reusable task prompt
    task = ChatPromptTemplate.from_messages([("human", "{task}")])

    # Combine
    full_prompt = persona + task

    # Test different combinations
    model = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
    chain = full_prompt | model

    # As a pirate
    response = chain.invoke(
        {
            "role": "pirate captain",
            "tone": "adventurous",
            "task": "Tell me about your ship",
        }
    )
    print(f"Pirate: {response.content[:100]}...")

    # As a scientist
    response = chain.invoke(
        {
            "role": "scientist",
            "tone": "precise and academic",
            "task": "Explain photosynthesis",
        }
    )
    print(f"\nScientist: {response.content[:100]}...")

## Run

The original `__main__` guard, kept verbatim. Jupyter sets `__name__` to `"__main__"`, so this cell runs as-is. Uncomment a line to run that demo.

In [7]:
# ============ RUN =========================================================
if __name__ == "__main__":
    print("=" * 50)
    print("Demo 1: Basic Templates")
    print("=" * 50)
    demo_basic_templates()

    print("\n" + "=" * 50)
    print("Demo 2: Message Types")
    print("=" * 50)
    demo_message_types()

    print("\n" + "=" * 50)
    print("Demo 3: MessagesPlaceholder")
    print("=" * 50)
    demo_messages_placeholder()

    print("\n" + "=" * 50)
    print("Demo 4: Few-Shot")
    print("=" * 50)
    demo_few_shot()

    print("\n" + "=" * 50)
    print("Demo 5: Prompt Composition")
    print("=" * 50)
    demo_prompt_composition()

Demo 1: Basic Templates
Simple template:
  [HumanMessage(content="Translate 'Hello, world!' to French", additional_kwargs={}, response_metadata={})]

Multi-message template:
  SystemMessage: You are a translator. Be concise.
  HumanMessage: Translate 'Good morning' to Japanese

Demo 2: Message Types
Conversation result: 35.

Demo 3: MessagesPlaceholder
With history placeholder:
  SystemMessage: You are a helpful assistant....
  HumanMessage: My name is Paulo...
  AIMessage: Nice to meet you, Paulo!...
  HumanMessage: What's my name?...

Response: Your name is Paulo.

Demo 4: Few-Shot
Few-shot result: The opposite of 'bright' is 'dim'.

Demo 5: Prompt Composition
Pirate: Ahoy, matey! Let me regale ye with tales of me trusty vessel, the *Sea Serpent*! A grand ship she be...

Scientist: Photosynthesis is a biochemical process through which autotrophic organisms, primarily plants, algae...


## Summary

Defined in this notebook:

- `demo_basic_templates()`
- `demo_message_types()`
- `demo_messages_placeholder()`
- `demo_few_shot()`
- `demo_prompt_composition()`